# NB1 — Synthetic Data Generator

Generates 84k rows of realistic inverter+relay telemetry across 5 degradation scenarios.

**Key insight:** Arc damage per switch scales with I² — a 15A AC inrush causes **225×** more wear than a 1A switch.

**Outputs:** `aria_data/combined_training_data.csv`, per-scenario CSVs, `metadata.json`, health curve plot.

**ARIA — Relay Intelligence · APOGEE'26 · BITS Pilani · Luminous Power Technologies**

## Installation

In [ ]:
!pip install numpy pandas matplotlib

## Setup

In [ ]:
"""
ARIA — Adaptive Relay Intelligence & Anomaly System
Notebook 1: Synthetic Data Generator

Generates realistic inverter + relay telemetry data for:
- Training the LSTM / LightGBM models (Notebook 2)
- Testing the physics-based Health Index model
- Simulating different degradation scenarios

RAG stack note: plug your own RAG embedder in notebook_rag.py later.
Multimodal note: image-based relay degradation labels added as a column
so your image-understanding pipeline can later align with sensor data.
"""

import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta
import json
import os

## Section

In [ ]:
# 0. GLOBAL SEED & CONFIG

## Section

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

OUTPUT_DIR = "aria_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Relay datasheet constants (typical Omron G2R / Songle SRD class)
RELAY_CONFIG = {
    "rated_cycles_at_rated_load": 100_000,   # at 10A, 250VAC
    "rated_switching_current_A": 10.0,
    "max_temp_C": 70.0,
    "coil_voltage_V": 12.0,
    "contact_resistance_new_mOhm": 50.0,     # milliohms when new
    "contact_resistance_fail_mOhm": 500.0,   # milliohms at failure
    "arc_constant_k": 1.2e-9,               # empirical, tunable
}

# Scenario definitions — controls what kind of degradation story we tell
SCENARIOS = {
    "healthy_household":   {"ac_starts_per_day": 0,  "load_pct_mean": 35, "power_cuts_per_day": 1,  "temp_mean_C": 32, "dust_factor": 1.0},
    "heavy_ac_user":       {"ac_starts_per_day": 8,  "load_pct_mean": 75, "power_cuts_per_day": 2,  "temp_mean_C": 38, "dust_factor": 1.2},
    "frequent_cuts":       {"ac_starts_per_day": 2,  "load_pct_mean": 45, "power_cuts_per_day": 12, "temp_mean_C": 33, "dust_factor": 1.1},
    "hot_environment":     {"ac_starts_per_day": 3,  "load_pct_mean": 55, "power_cuts_per_day": 3,  "temp_mean_C": 52, "dust_factor": 1.5},
    "industrial_ups":      {"ac_starts_per_day": 20, "load_pct_mean": 85, "power_cuts_per_day": 5,  "temp_mean_C": 45, "dust_factor": 1.8},
}

## Section

In [ ]:
# 1. PHYSICS MODEL
# Core insight: damage per switch scales with I²
# A relay switching at 15A (AC inrush) does 225× more damage than at 1A

## Section

In [ ]:
def arc_damage_per_switch(I_switch_A: float, V_relay_V: float, bounce_ms: float) -> float:
    """
    Energy dissipated in contact arc per switching event.
    E_arc ∝ k × I² × V × t_bounce
    Returns: dimensionless wear units
    """
    k = RELAY_CONFIG["arc_constant_k"]
    return k * (I_switch_A ** 2) * V_relay_V * bounce_ms


def arrhenius_temp_factor(temp_C: float, ref_temp_C: float = 25.0) -> float:
    """
    Arrhenius acceleration factor for thermal degradation.
    Every 10°C above reference roughly doubles degradation rate.
    Factor > 1 means faster wear than rated.
    """
    Ea = 0.7      # activation energy in eV (typical electromechanical)
    k_B = 8.617e-5  # Boltzmann constant in eV/K
    T = temp_C + 273.15
    T_ref = ref_temp_C + 273.15
    return np.exp(Ea / k_B * (1/T_ref - 1/T))


def health_index(cumulative_wear: float, max_rated_wear: float) -> float:
    """
    Health Index: 100 = new, 0 = failed.
    Non-linear decay — accelerates as contacts degrade.
    """
    ratio = cumulative_wear / max_rated_wear
    # Sigmoid-like decay: slow at first, accelerates past 60% wear
    hi = 100 * (1 - ratio) * (1 + 0.3 * np.sin(np.pi * ratio))
    return float(np.clip(hi, 0, 100))


def contact_resistance_mOhm(cumulative_wear: float, max_rated_wear: float) -> float:
    """
    Contact resistance increases as arc erosion builds up oxide layer.
    Interpolates from new (50mΩ) to failure (500mΩ).
    """
    ratio = min(cumulative_wear / max_rated_wear, 1.0)
    r_new  = RELAY_CONFIG["contact_resistance_new_mOhm"]
    r_fail = RELAY_CONFIG["contact_resistance_fail_mOhm"]
    return r_new + (r_fail - r_new) * (ratio ** 1.5)


def bounce_time_ms(cumulative_wear: float, max_rated_wear: float,
                   base_bounce_ms: float = 2.0) -> float:
    """
    Contact bounce duration increases as spring mechanism wears.
    Fresh relay: ~2ms bounce. Degraded: up to 8ms.
    This is a key observable for the edge firmware to measure.
    """
    ratio = min(cumulative_wear / max_rated_wear, 1.0)
    degradation_factor = 1 + 3 * (ratio ** 2)
    noise = np.random.normal(0, 0.2)
    return max(0.5, base_bounce_ms * degradation_factor + noise)

## Section

In [ ]:
# 2. MODBUS REGISTER SIMULATOR
# Mirrors the register map from the Luminous PDF
# Registers: 3002–3059 + custom relay registers

## Section

In [ ]:
def simulate_modbus_snapshot(
    timestamp: datetime,
    scenario: dict,
    cumulative_wear: float,
    max_rated_wear: float,
    is_power_cut: bool,
    is_ac_start: bool,
) -> dict:
    """
    Returns one row of inverter telemetry as a dict.
    Keys match MODBUS register names from Luminous PDF.
    """
    temp_C = np.random.normal(scenario["temp_mean_C"], 3.0)
    temp_C = np.clip(temp_C, 20, 75)

    load_pct = np.random.normal(scenario["load_pct_mean"], 8.0)
    load_pct = np.clip(load_pct, 10, 100)

    # Inrush current spikes on AC motor start (5–8× rated)
    if is_ac_start:
        inverter_current_A = np.random.uniform(12, 18)   # inrush
    else:
        inverter_current_A = load_pct / 100 * 10 + np.random.normal(0, 0.3)

    mains_voltage = 0 if is_power_cut else np.random.normal(220, 8)
    mains_ok = 0 if is_power_cut else 1

    # System state: 1=standby/mains, 2=inverter mode, 3=charging
    if is_power_cut:
        system_state = 2
    elif load_pct > 80:
        system_state = 3
    else:
        system_state = 1

    # Relay-specific computed fields (custom registers — added at end of map)
    hi = health_index(cumulative_wear, max_rated_wear)
    bounce_ms = bounce_time_ms(cumulative_wear, max_rated_wear)
    contact_r = contact_resistance_mOhm(cumulative_wear, max_rated_wear)
    rul_days = estimate_rul_days(cumulative_wear, max_rated_wear, scenario)

    # Visual degradation label for your multimodal RAG pipeline
    # Aligns sensor data with relay contact photographs
    if hi > 80:
        visual_label = "new"
    elif hi > 60:
        visual_label = "mild_pitting"
    elif hi > 35:
        visual_label = "moderate_erosion"
    elif hi > 15:
        visual_label = "severe_erosion"
    else:
        visual_label = "pre_failure"

    return {
        # ── Standard MODBUS registers (from Luminous PDF) ──
        "timestamp":              timestamp.isoformat(),
        "reg_3002_discharge_A":   round(max(0, inverter_current_A), 2),
        "reg_3003_charging_A":    round(np.random.uniform(0, 5) if system_state == 3 else 0, 2),
        "reg_3004_mains_V":       round(max(0, mains_voltage), 1),
        "reg_3007_trip_code":     0,   # 0 = no fault
        "reg_3011_switch":        1 if is_power_cut else 0,
        "reg_3012_mains_ok":      mains_ok,
        "reg_3019_temp_C":        round(temp_C, 1),
        "reg_3020_system_state":  system_state,
        "reg_3023_high_charge":   1 if load_pct > 85 else 0,
        "reg_3045_inverter_state":2 if is_power_cut else 1,
        "reg_3046_charging_state":1 if system_state == 3 else 0,
        "reg_3052_grid_CT_A":     round(np.random.normal(5, 0.5), 2),
        "reg_3053_inv_voltage_V": round(np.random.normal(230, 3), 1),
        "reg_3054_inv_current_A": round(max(0, inverter_current_A), 2),
        "reg_3058_grid_freq_Hz":  round(np.random.normal(50, 0.1), 2),
        "reg_3059_overload":      1 if inverter_current_A > 14 else 0,
        "reg_3060_load_pct":      round(load_pct, 1),

        # ── Custom relay registers (added at end of map as per PDF note) ──
        "reg_4001_relay_switch_count":    int(cumulative_wear / RELAY_CONFIG["arc_constant_k"] / 10),
        "reg_4002_relay_health_index":    round(hi, 2),
        "reg_4003_contact_bounce_ms":     round(bounce_ms, 3),
        "reg_4004_contact_resist_mOhm":   round(contact_r, 1),
        "reg_4005_arc_energy_this_switch":round(
            arc_damage_per_switch(inverter_current_A, 12, bounce_ms) * 1e9, 4
        ) if is_power_cut or is_ac_start else 0.0,
        "reg_4006_rul_days":              round(rul_days, 1),

        # ── Derived features (for ML training, not sent over MODBUS) ──
        "is_power_cut":       int(is_power_cut),
        "is_ac_start":        int(is_ac_start),
        "cumulative_wear":    round(cumulative_wear, 6),
        "temp_factor":        round(arrhenius_temp_factor(temp_C), 4),
        "visual_label":       visual_label,    # for multimodal RAG alignment
        "scenario":           "",              # filled by caller
        "alert_level":        get_alert_level(hi),
    }


def get_alert_level(hi: float) -> str:
    if hi > 60:  return "green"
    if hi > 30:  return "yellow"
    if hi > 10:  return "orange"
    return "red"


def estimate_rul_days(cumulative_wear: float, max_rated_wear: float,
                      scenario: dict) -> float:
    """
    Simple linear RUL estimate based on current wear rate.
    The LSTM in Notebook 2 will produce a much better non-linear version.
    """
    remaining_wear = max(0, max_rated_wear - cumulative_wear)
    # Estimate daily wear from scenario parameters
    daily_switches = scenario["power_cuts_per_day"] + scenario["ac_starts_per_day"] * 2
    avg_current = scenario["load_pct_mean"] / 100 * 10
    daily_wear = daily_switches * arc_damage_per_switch(avg_current, 12, 3.0)
    daily_wear *= arrhenius_temp_factor(scenario["temp_mean_C"])
    daily_wear *= scenario["dust_factor"]
    if daily_wear <= 0:
        return 9999.0
    return min(remaining_wear / daily_wear, 9999.0)

## Section

In [ ]:
# 3. MAIN GENERATOR

## Section

In [ ]:
def generate_scenario(
    scenario_name: str,
    scenario: dict,
    n_days: int = 365,
    samples_per_day: int = 48,   # every 30 min
) -> pd.DataFrame:
    """
    Generates n_days × samples_per_day rows of telemetry for one relay.
    Relay degrades over time following the physics model.
    Returns a DataFrame ready for ML training.
    """
    print(f"  Generating scenario: {scenario_name} ({n_days} days × {samples_per_day} samples/day)")

    max_rated_wear = (
        RELAY_CONFIG["rated_cycles_at_rated_load"]
        * arc_damage_per_switch(
            RELAY_CONFIG["rated_switching_current_A"],
            12.0,
            2.0
        )
    )

    records = []
    cumulative_wear = 0.0
    start_time = datetime(2024, 1, 1, 0, 0, 0)

    for day in range(n_days):
        # Decide today's events
        n_cuts = np.random.poisson(scenario["power_cuts_per_day"])
        n_ac   = np.random.poisson(scenario["ac_starts_per_day"])
        cut_samples = set(random.sample(range(samples_per_day), min(n_cuts, samples_per_day)))
        ac_samples  = set(random.sample(range(samples_per_day), min(n_ac,  samples_per_day)))

        for s in range(samples_per_day):
            timestamp = start_time + timedelta(days=day, minutes=30*s)
            is_cut    = s in cut_samples
            is_ac     = s in ac_samples

            # Compute this sample's switching current
            if is_cut or is_ac:
                I_switch = np.random.uniform(8, 18) if is_ac else np.random.uniform(4, 10)
                temp_C   = np.random.normal(scenario["temp_mean_C"], 3)
                bounce   = bounce_time_ms(cumulative_wear, max_rated_wear)
                damage   = arc_damage_per_switch(I_switch, 12.0, bounce)
                damage  *= arrhenius_temp_factor(temp_C)
                damage  *= scenario["dust_factor"]
                cumulative_wear += damage

            snap = simulate_modbus_snapshot(
                timestamp, scenario, cumulative_wear, max_rated_wear,
                is_cut, is_ac
            )
            snap["scenario"] = scenario_name
            records.append(snap)

            # Stop early if relay has failed
            if cumulative_wear >= max_rated_wear:
                print(f"    → Relay failed on day {day}, sample {s}")
                df = pd.DataFrame(records)
                return df

    return pd.DataFrame(records)

## Section

In [ ]:
# 4. RUN ALL SCENARIOS + SAVE

## Section

In [ ]:
def main():
    print("=" * 60)
    print("ARIA — Notebook 1: Synthetic Data Generator")
    print("=" * 60)

    all_dfs = []

    for name, scenario in SCENARIOS.items():
        df = generate_scenario(name, scenario, n_days=400, samples_per_day=48)
        out_path = os.path.join(OUTPUT_DIR, f"{name}.csv")
        df.to_csv(out_path, index=False)
        print(f"  Saved {len(df):,} rows → {out_path}")
        all_dfs.append(df)

    # Combined dataset for training
    combined = pd.concat(all_dfs, ignore_index=True)
    combined_path = os.path.join(OUTPUT_DIR, "combined_training_data.csv")
    combined.to_csv(combined_path, index=False)

    print("\n" + "=" * 60)
    print(f"Combined dataset: {len(combined):,} rows × {len(combined.columns)} columns")
    print(f"Scenarios: {combined['scenario'].value_counts().to_dict()}")
    print(f"Alert distribution:\n{combined['alert_level'].value_counts()}")
    print(f"Visual labels (for multimodal RAG):\n{combined['visual_label'].value_counts()}")
    print("=" * 60)
    print(f"\nAll files saved to: ./{OUTPUT_DIR}/")
    print("Next step → Notebook 2: ML model training (LSTM + LightGBM)")

    # Save relay config and scenario metadata as JSON (for RAG knowledge base)
    meta = {
        "relay_config": RELAY_CONFIG,
        "scenarios": SCENARIOS,
        "columns": list(combined.columns),
        "modbus_map": {
            "3002": "DISCHARGE_CURRENT",
            "3003": "CHARGING_CURRENT",
            "3004": "MAINS_VOLTAGE",
            "3007": "TRIP_CODE",
            "3011": "SWITCH",
            "3012": "MAINS_OK_FLAG",
            "3019": "TEMPERATURE",
            "3020": "SYSTEM_STATE",
            "3023": "HIGH_CHARGING_MODE",
            "3045": "INVERTER_STATE",
            "3046": "CHARGING_STATE",
            "3052": "GRID_CT_CURRENT",
            "3053": "INV_VOLTAGE",
            "3054": "INVERTER_CURRENT",
            "3058": "GRID_FREQUENCY",
            "3059": "LINE_CURRENT_OVERLOAD",
            "4001": "RELAY_SWITCH_COUNT",
            "4002": "RELAY_HEALTH_INDEX",
            "4003": "CONTACT_BOUNCE_MS",
            "4004": "CONTACT_RESISTANCE_mOHM",
            "4005": "ARC_ENERGY_THIS_SWITCH",
            "4006": "RUL_DAYS",
        }
    }
    with open(os.path.join(OUTPUT_DIR, "metadata.json"), "w") as f:
        json.dump(meta, f, indent=2)

    return combined

## Section

In [ ]:
# 5. QUICK VALIDATION PLOT (optional, needs matplotlib)

## Section

In [ ]:
def plot_health_curves(output_dir: str = OUTPUT_DIR):
    try:
        import matplotlib.pyplot as plt

        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        fig.suptitle("ARIA — Relay Health Index Over Time (All Scenarios)", fontsize=14)

        for ax, (name, _) in zip(axes.flat, SCENARIOS.items()):
            path = os.path.join(output_dir, f"{name}.csv")
            df = pd.read_csv(path)
            df["timestamp"] = pd.to_datetime(df["timestamp"])
            daily = df.groupby(df["timestamp"].dt.date)["reg_4002_relay_health_index"].mean()

            ax.plot(daily.values, color="#1D9E75", linewidth=1.5)
            ax.axhline(30, color="#BA7517", linestyle="--", linewidth=0.8, label="Warn (30)")
            ax.axhline(10, color="#E24B4A", linestyle="--", linewidth=0.8, label="Critical (10)")
            ax.set_title(name.replace("_", " ").title(), fontsize=10)
            ax.set_xlabel("Days")
            ax.set_ylabel("Health Index")
            ax.set_ylim(0, 105)
            ax.legend(fontsize=7)
            ax.grid(True, alpha=0.3)

        # Hide unused subplot if scenarios < 6
        for ax in axes.flat[len(SCENARIOS):]:
            ax.set_visible(False)

        plt.tight_layout()
        plot_path = os.path.join(output_dir, "health_curves.png")
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        print(f"Plot saved → {plot_path}")
        plt.show()

    except ImportError:
        print("matplotlib not installed — skipping plot. Run: pip install matplotlib")


if __name__ == "__main__":
    df = main()
    plot_health_curves()

## Run Everything

In [ ]:
main()

# Plot health curves
import matplotlib.pyplot as plt
fig,axes=plt.subplots(2,3,figsize=(15,8))
for ax,(name,_) in zip(axes.flat, SCENARIOS.items()):
    df=pd.read_csv(f'aria_data/{name}.csv')
    df['timestamp']=pd.to_datetime(df['timestamp'])
    daily=df.groupby(df['timestamp'].dt.date)['reg_4002_relay_health_index'].mean()
    ax.plot(daily.values,color='#1D9E75',linewidth=1.5)
    ax.axhline(30,color='#BA7517',linestyle='--',linewidth=0.8,label='Warn')
    ax.axhline(10,color='#E24B4A',linestyle='--',linewidth=0.8,label='Critical')
    ax.set_title(name.replace('_',' ').title(),fontsize=10)
    ax.set_ylim(0,105); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
for ax in axes.flat[len(SCENARIOS):]: ax.set_visible(False)
plt.tight_layout(); plt.savefig('aria_data/health_curves.png',dpi=150); plt.show()